# 🧪 [Colab 실습] 프루닝 & 지식 증류 미니랩 — 경량화 3종 세트 완성

**온디바이스 AI 프로그래밍 · 「양자화 미니랩」 자매편 — Day 2 경량화 이론의 나머지 두 무기**

| 항목 | 내용 |
| --- | --- |
| 실습 목표 | ① 비구조적 프루닝이 왜 "실속 없는 다이어트"인지 실측 ② 구조적 프루닝의 "연쇄 수술" 직접 집도 ③ 지식 증류로 잘린 모델 회복 ④ 3종 조합으로 교안 목표 "10× 압축" 달성 |
| 환경 | Google Colab — **GPU 권장**(학습 2~3분), CPU도 축소 모드로 완주 가능 |
| 핵심 API | `torch.nn.utils.prune` · KD loss 직접 구현 · `torch.ao.quantization`(Part 5 조합) |

## 이 실습의 위치 — 교안 Day 2 「경량화 4가지 무기」에서

```text
Raw PyTorch Model ──┬── 양자화 (Quantization)      ← 양자화 미니랩에서 완료 ✅
                    ├── 프루닝 (Pruning)           ← ★ 오늘 Part 2~3
                    ├── 지식 증류 (KD)             ← ★ 오늘 Part 4
                    └── XWN (디퍼아이 독자 기법)    ← Day 2 본 실습
                         ↓
                  Optimized Model  (오늘 Part 5에서 3종 조합으로 체험)
```

## 실습 로드맵

| Part | 주제 | 교안 대응 |
| --- | --- | --- |
| 1 | 준비 — 모델·데이터·FP32 기준선 | — |
| 2 | ★ 비구조적 프루닝의 배신 (90% sparsity의 진실) | 프루닝 심화: Unstructured |
| 3 | ★ 구조적 프루닝 — 연쇄 수술과 재학습 | 프루닝 심화: Structured |
| 4 | ★ 지식 증류 — Teacher의 dark knowledge | KD 실전 가이드 (T=5, α=0.8) |
| 5 | 3종 조합 — 프루닝+KD+양자화 = 10× 압축 | Day 2 Before/After |
| 6 | 리포트 & Day 2 연결 | — |


---
# Part 0. 환경 준비

In [ ]:
import copy, io, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
import torch.ao.quantization as tq
import matplotlib.pyplot as plt

try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"], capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
except Exception:
    pass
plt.rc("axes", unicode_minus=False)

torch.manual_seed(42); random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.quantized.engine = "fbgemm"
print("torch:", torch.__version__, "| 학습 장치:", device)

---
# Part 1. 준비 — 모델·데이터·FP32 기준선

### Step 1-1. CIFAR-10 로드 (양자화 미니랩과 동일)

In [ ]:
import torchvision
import torchvision.transforms as T

tf = T.Compose([T.ToTensor(), T.Normalize((0.49, 0.48, 0.45), (0.25, 0.24, 0.26))])
train_full = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf)

n_train = len(train_full) if device == "cuda" else 10_000
train_set = torch.utils.data.Subset(train_full, range(n_train))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)
print(f"학습 {len(train_set):,}장 / 평가 {len(test_set):,}장")

### Step 1-2. 모델 정의 — 채널 수를 인자로 받는 설계 (복선!)

양자화 미니랩의 MiniCNN과 같지만 한 가지가 다릅니다: **채널 수 `chs`를 생성자 인자로** 받습니다.
Part 3에서 채널을 잘라낸 "작은 쌍둥이 모델"을 만들 때 이 설계가 빛을 발합니다.
QuantStub도 유지합니다 — Part 5에서 양자화까지 조합할 것이기 때문입니다.

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, cin, cout, stride=1):
        super().__init__(nn.Conv2d(cin, cout, 3, stride, 1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU())

class MiniCNN(nn.Module):
    def __init__(self, chs=(32, 64, 128), num_classes=10):    # ★ 채널 수 파라미터화
        super().__init__()
        self.quant = tq.QuantStub()
        self.b1 = ConvBNReLU(3,      chs[0])
        self.b2 = ConvBNReLU(chs[0], chs[1], stride=2)
        self.b3 = ConvBNReLU(chs[1], chs[2], stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(chs[2], num_classes)
        self.dequant = tq.DeQuantStub()
    def forward(self, x):
        x = self.quant(x)
        x = self.b3(self.b2(self.b1(x)))
        x = self.pool(x).flatten(1)
        return self.dequant(self.fc(x))
    def fuse_model(self):
        for n in ["b1", "b2", "b3"]:
            tq.fuse_modules(getattr(self, n), [["0", "1", "2"]], inplace=True)

model = MiniCNN().to(device)
print(f"파라미터: {sum(p.numel() for p in model.parameters()):,}개  (chs={tuple([32,64,128])})")

### Step 1-3. 학습 + FP32 기준선 (양자화 미니랩과 동일 패턴)

In [ ]:
def evaluate(m, loader, dev="cpu"):
    m.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            out = m(x.to(dev))
            correct += (out.argmax(1).cpu() == y).sum().item(); total += y.numel()
    return correct / total * 100

def model_size_kb(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1024

def bench_ms(m, batch=16, n=30):
    xb = torch.randn(batch, 3, 32, 32); m.eval()
    with torch.no_grad():
        for _ in range(5): m(xb)
        t0 = time.perf_counter()
        for _ in range(n): m(xb)
    return (time.perf_counter() - t0) / n * 1000

EPOCHS = 3 if device == "cuda" else 1
opt  = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(train_loader))
crit = nn.CrossEntropyLoss()

model.train()
for ep in range(EPOCHS):
    t0 = time.time()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step(); sch.step()
    print(f"epoch {ep+1}/{EPOCHS}  loss {loss.item():.3f}  ({time.time()-t0:.0f}s)")

model = model.cpu().eval()
fp32_acc, fp32_size, fp32_ms = evaluate(model, test_loader), model_size_kb(model), bench_ms(model)
print(f"\n═══ FP32 기준선 ═══\n정확도 {fp32_acc:.2f}% | 크기 {fp32_size:.0f}KB | 지연 {fp32_ms:.1f}ms")

---
# Part 2. ★ 비구조적 프루닝의 배신 — 90% 다이어트인데 몸무게 그대로?

교안 프루닝 심화의 핵심 대비를 실측합니다.

> **Unstructured**: "정확도 손실 매우 작음(90% sparsity까지), 이론 연산량 90% 감소" —
> 그러나 "**일반 NPU에서 속도 향상 X, sparse matrix 특수 HW 필요**"

### Step 2-1. 90% 프루닝 실행 — weight의 90%를 0으로

`global_unstructured`는 모델 전체 weight를 한 줄로 세워 **크기가 작은 순서대로 90%를 0으로** 만듭니다
(L1 기준). 레이어별로 균등하게가 아니라 전역에서 고르는 것이 포인트입니다.

In [ ]:
m_unstr = copy.deepcopy(model)
targets = [(m_unstr.b1[0], "weight"), (m_unstr.b2[0], "weight"),
           (m_unstr.b3[0], "weight"), (m_unstr.fc, "weight")]

prune.global_unstructured(targets, pruning_method=prune.L1Unstructured, amount=0.9)

total = zeros = 0
print(f"{'레이어':<8}{'전체':>10}{'0의 개수':>10}{'sparsity':>10}")
for mod, name in targets:
    w = getattr(mod, name)
    z = (w == 0).sum().item()
    total += w.numel(); zeros += z
    lname = "fc" if isinstance(mod, nn.Linear) else f"conv{targets.index((mod,name))+1}"
    print(f"{lname:<8}{w.numel():>10,}{z:>10,}{z/w.numel()*100:>9.1f}%")
print(f"{'전체':<8}{total:>10,}{zeros:>10,}{zeros/total*100:>9.1f}%")
print("\n💡 전역 L1 기준이라 레이어마다 sparsity가 다릅니다 — 중요한 레이어는 덜 잘렸습니다.")

### Step 2-2. sparsity별 정확도 곡선 — "정확도는 정말 버틴다"

재학습 없이 30~90%까지 잘라가며 정확도를 잽니다.
교안의 "정확도 손실 매우 작음"이 어디까지 버티는지 직접 확인하세요.

In [ ]:
ratios, accs = [0.0, 0.3, 0.5, 0.7, 0.9], []
for r in ratios:
    mt = copy.deepcopy(model)
    if r > 0:
        tg = [(mt.b1[0], "weight"), (mt.b2[0], "weight"), (mt.b3[0], "weight"), (mt.fc, "weight")]
        prune.global_unstructured(tg, pruning_method=prune.L1Unstructured, amount=r)
    accs.append(evaluate(mt, test_loader))

plt.figure(figsize=(7, 3.5))
plt.plot([r*100 for r in ratios], accs, "o-", lw=2)
plt.axhline(fp32_acc, ls="--", c="gray", label=f"FP32 기준 {fp32_acc:.1f}%")
plt.xlabel("sparsity (%)"); plt.ylabel("정확도 (%)")
plt.title("비구조적 프루닝 — 재학습 없이 어디까지 버티나")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

for r, a in zip(ratios, accs):
    print(f"sparsity {r*100:>3.0f}% → 정확도 {a:.2f}%  ({a-fp32_acc:+.2f}%p)")

### Step 2-3. 그런데... 크기와 속도는? — 배신의 순간

`prune.remove()`로 마스크를 weight에 영구 반영한 뒤, 파일 크기와 추론 속도를 잽니다.
**90%가 0인데 무엇이 달라졌을까요?**

In [ ]:
for mod, name in targets:
    prune.remove(mod, name)          # 마스크 → weight에 영구 반영 (weight_orig 제거)

u_size, u_ms = model_size_kb(m_unstr), bench_ms(m_unstr)
u_acc = evaluate(m_unstr, test_loader)

print(f"{'':16}{'FP32 원본':>10}{'비구조 90%':>11}")
print(f"{'정확도':<14}{fp32_acc:>9.2f}%{u_acc:>10.2f}%")
print(f"{'크기':<15}{fp32_size:>8.0f}KB{u_size:>9.0f}KB   ← 그대로!")
print(f"{'지연':<15}{fp32_ms:>8.1f}ms{u_ms:>9.1f}ms   ← 그대로! (±측정 오차)")
print()
print("💥 90%가 0이어도: 0도 float32로 '저장'되고, 0×x도 곱셈 1회로 '연산'됩니다.")
print("   dense 텐서의 모양(shape)이 그대로인 한, 저장 장치도 MAC 배열도 0을 특별 취급하지 않습니다.")
print("   → 교안: '일반 NPU에서 속도 향상 X, sparse matrix 특수 HW 필요'")

### Step 2-4. sparse 형식으로 저장하면 되지 않나? — 인덱스의 역습

"0을 빼고 (위치, 값)만 저장"하는 sparse(COO) 형식으로 저장해 봅니다.
90% sparsity면 크기가 1/10이 될 것 같지만...

In [ ]:
sparse_sd = {}
for k, v in m_unstr.state_dict().items():
    if v.dim() >= 2 and (v == 0).float().mean() > 0.5:
        sparse_sd[k] = v.to_sparse()          # (indices + values)로 저장
    else:
        sparse_sd[k] = v
buf = io.BytesIO(); torch.save(sparse_sd, buf)
sparse_kb = buf.getbuffer().nbytes / 1024

print(f"dense 저장  : {u_size:.0f} KB")
print(f"sparse 저장 : {sparse_kb:.0f} KB   (기대: 1/10 수준... 실제: {u_size/sparse_kb:.1f}배 감소뿐)")
print()
print("💡 살아남은 10%의 값마다 '몇 행 몇 열인지' 인덱스(int64!)를 함께 저장해야 하기 때문입니다.")
print("   값 1개(4B)에 좌표가 따라붙으니 압축 효율이 무너집니다 — 연산은 말할 것도 없고요.")
print()
print("═══ Part 2 결론 ═══")
print("비구조적 프루닝: 정확도 방어력 최고, 그러나 크기·속도 실익은 특수 HW 없이는 없다.")
print("→ 그래서 Part 3, '아예 채널을 도려내는' 구조적 프루닝이 필요합니다.")

> **✅ Part 2 확인**
> - [ ] 90% sparsity에도 파일 크기·추론 속도가 불변임을 실측했다
> - [ ] sparse 저장조차 인덱스 오버헤드로 큰 이득이 없음을 확인했다
> - [ ] "이론 연산량 감소 ≠ 실속 가속"을 설명할 수 있다 (가상 NPU 실습: MAC 배열은 0을 건너뛰지 않는다)

---
# Part 3. ★ 구조적 프루닝 — 채널을 통째로 도려내는 연쇄 수술

교안: "*채널/필터 단위 삭제 → **모든 NPU에서 실속 가속**, 모델 파일 크기 그대로 감소.
단점: 정확도 손실 크다(보상 필요) → Fine-tuning 재학습 필수*"

### Step 3-1. 수술 계획 — 어느 채널을 자를 것인가 (L1 중요도)

각 Conv의 출력 채널별 |weight| 합(L1-norm)을 중요도로 삼습니다.
값이 작은 채널 = "출력에 기여가 적은 채널" = 절제 대상.

In [ ]:
def channel_importance(conv):
    """출력 채널별 L1-norm — (out, in, kh, kw)에서 out 축만 남기고 절대값 합"""
    return conv.weight.detach().abs().sum(dim=(1, 2, 3))

imp = channel_importance(model.b2[0])                 # b2: 64채널
keep_n = len(imp) // 2
thresh = torch.topk(imp, keep_n).values.min()

colors = ["#4c72b0" if v >= thresh else "#c44e52" for v in imp]
plt.figure(figsize=(10, 3.2))
plt.bar(range(len(imp)), imp.tolist(), color=colors)
plt.axhline(thresh.item(), ls="--", c="k", lw=1, label=f"생존선 (상위 {keep_n}개)")
plt.xlabel("b2 출력 채널"); plt.ylabel("L1 중요도")
plt.title("파란색=생존, 빨간색=절제 대상 — 채널 중요도는 생각보다 불균등하다")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

print(f"중요도 최대/최소 비율: {(imp.max()/imp.min()).item():.1f}배")

### Step 3-2. 연쇄 수술 집도 — 구조적 프루닝이 어려운 진짜 이유

b2의 출력 채널 하나를 자르면 무슨 일이 생기는지 따라가 봅시다:

```text
b2 Conv의 출력 채널 32번 절제
  → b2 BN의 32번 통계(γ, β, mean, var)도 함께 절제         (같은 채널이므로)
  → b3 Conv의 "입력" 채널 32번도 절제                        (사라진 입력을 받던 가중치)
  → b3를 자르면 fc의 입력 특성도 절제                        (연쇄는 끝까지 전파!)
```

**한 채널의 절제가 앞뒤 레이어로 전파**됩니다 — 이것이 "연쇄 수술"이고,
비구조적 프루닝(마스크 한 장)과 난이도가 다른 이유입니다.
Part 1에서 `chs`를 파라미터로 만든 복선을 여기서 회수합니다: 작은 쌍둥이 모델을 만들어
살아남은 채널의 weight만 이식합니다.

In [ ]:
def structured_prune(base, keep=0.5):
    """채널 keep 비율만 남기는 연쇄 수술. 반환: 작은 모델(slim), 생존 채널 인덱스들"""
    chs    = [base.b1[0].out_channels, base.b2[0].out_channels, base.b3[0].out_channels]
    keep_n = [max(1, int(c * keep)) for c in chs]

    # ① 수술 계획: 블록별 생존 채널 결정
    idxs = []
    for blk, k in zip([base.b1, base.b2, base.b3], keep_n):
        imp = channel_importance(blk[0])
        idxs.append(torch.sort(torch.topk(imp, k).indices).values)

    # ② 작은 쌍둥이 모델 생성 (Part 1의 chs 파라미터가 여기서 활약!)
    slim = MiniCNN(chs=tuple(keep_n))

    # ③ 장기 이식: 살아남은 채널의 weight만 복사
    prev = None                                    # 직전 블록의 생존 인덱스 (in-채널 동기화용)
    for blk_o, blk_s, idx in zip([base.b1, base.b2, base.b3],
                                 [slim.b1, slim.b2, slim.b3], idxs):
        w = blk_o[0].weight.detach()[idx]          # out 채널 슬라이스
        if prev is not None:
            w = w[:, prev]                         # ★ in 채널 동기화 — 연쇄의 핵심!
        blk_s[0].weight.data.copy_(w)
        for attr in ["weight", "bias", "running_mean", "running_var"]:
            getattr(blk_s[1], attr).data.copy_(getattr(blk_o[1], attr).detach()[idx])
        prev = idx
    slim.fc.weight.data.copy_(base.fc.weight.detach()[:, idxs[2]])   # fc 입력도 연쇄!
    slim.fc.bias.data.copy_(base.fc.bias.detach())
    return slim, idxs

slim_raw, kept = structured_prune(model, keep=0.5)
slim_raw.eval()
print("수술 완료 — 채널 구성:", [len(i) for i in kept], "(원본 [32, 64, 128]의 절반)")
print("forward 동작 확인:", tuple(slim_raw(torch.randn(2, 3, 32, 32)).shape))

### Step 3-3. 수술 직후 검진 — 작아졌고 빨라졌지만, 아프다

교안 예고: 구조적 프루닝은 "*정확도 손실 크다 → Fine-tuning 재학습이 필수*".
수술 직후 3축을 재 봅시다.

In [ ]:
s_size, s_ms = model_size_kb(slim_raw), bench_ms(slim_raw)
s_acc_raw = evaluate(slim_raw, test_loader)

print(f"{'':16}{'FP32 원본':>10}{'구조 50% (수술직후)':>18}")
print(f"{'정확도':<14}{fp32_acc:>9.2f}%{s_acc_raw:>17.2f}%   ← 급락! (재학습 전)")
print(f"{'크기':<15}{fp32_size:>8.0f}KB{s_size:>16.0f}KB   ← 진짜 줄었다 ({fp32_size/s_size:.1f}배)")
print(f"{'지연':<15}{fp32_ms:>8.1f}ms{s_ms:>16.1f}ms   ← 진짜 빨라졌다 ({fp32_ms/s_ms:.1f}배)")
print()
print("💡 비구조와 정반대: 크기·속도는 즉시 이득, 정확도는 재학습으로 갚아야 할 빚.")
print("   BlackSwan의 'Pruning Engine'과 '디퍼아이 전용 엔진 기반 재학습'(교안)이")
print("   바로 이 구조적 절제 + 재학습 루프를 하드웨어·SDK 수준에서 지원하는 것입니다.")

### Step 3-4. Fine-tuning으로 정확도 회복 (일반 CE 학습)

수술 직후 모델(`slim_raw`)은 Part 4의 공정 비교를 위해 보존하고,
사본을 만들어 CE loss로 재학습합니다. (Part 4에서 같은 출발점의 다른 사본을 KD로 학습해 대결!)

In [ ]:
def finetune(student, teacher=None, T=5.0, alpha=0.8, tag=""):
    '''teacher=None이면 일반 CE 학습, 주어지면 KD 학습 (Part 4에서 재사용)'''
    st = copy.deepcopy(student).to(device); st.train()
    opt = torch.optim.SGD(st.parameters(), lr=0.01, momentum=0.9)
    max_batches = None if device == "cuda" else 40          # CPU 축소 모드
    if teacher is not None:
        teacher = copy.deepcopy(teacher).to(device).eval()
    t0 = time.time()
    for bi, (x, y) in enumerate(train_loader):
        if max_batches and bi >= max_batches: break
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        s_logits = st(x)
        if teacher is None:
            loss = F.cross_entropy(s_logits, y)
        else:
            with torch.no_grad():
                t_logits = teacher(x)
            soft = F.kl_div(F.log_softmax(s_logits / T, dim=1),
                            F.softmax(t_logits / T, dim=1),
                            reduction="batchmean") * (T * T)   # T^2 스케일 (Part 4에서 설명)
            loss = alpha * soft + (1 - alpha) * F.cross_entropy(s_logits, y)
        loss.backward(); opt.step()
    st = st.cpu().eval()
    acc = evaluate(st, test_loader)
    print(f"[{tag}] 재학습 {time.time()-t0:.0f}s → 정확도 {acc:.2f}%")
    return st, acc

slim_ce, acc_ce = finetune(slim_raw, teacher=None, tag="CE fine-tune")
print(f"\n수술 직후 {s_acc_raw:.2f}% → CE 재학습 후 {acc_ce:.2f}%  (회복 {acc_ce-s_acc_raw:+.2f}%p)")

### Step 3-5. 중간 결산 — 비구조 90% vs 구조 50%

In [ ]:
print(f"{'':18}{'FP32':>9}{'비구조 90%':>11}{'구조 50%+CE':>13}")
print("-" * 54)
print(f"{'정확도 (%)':<16}{fp32_acc:>9.2f}{u_acc:>11.2f}{acc_ce:>13.2f}")
print(f"{'크기 (KB)':<16}{fp32_size:>9.0f}{u_size:>11.0f}{s_size:>13.0f}")
print(f"{'지연 (ms)':<16}{fp32_ms:>9.1f}{u_ms:>11.1f}{s_ms:>13.1f}")
print()
print("💡 표 하나에 교안의 결론이 다 담겼습니다:")
print("   비구조 = 정확도의 챔피언, 실속 0  /  구조 = 실속의 챔피언, 정확도는 재학습으로 보상")
print("   그리고 그 '보상'을 더 잘하는 방법이 Part 4의 지식 증류입니다 →")

> **✅ Part 3 확인**
> - [ ] 채널 절제가 BN·다음 Conv·fc까지 전파되는 "연쇄"를 코드로 구현했다
> - [ ] 구조적 프루닝의 즉시 이득(크기·속도)과 정확도 빚을 실측했다
> - [ ] 왜 최대 50% 수준이 현실적 한계인지(교안) 감을 잡았다 — keep을 0.25로 낮춰 직접 확인해 보세요 ✏️

---
# Part 4. ★ 지식 증류 — Teacher의 dark knowledge를 물려받기

교안 KD 실전 가이드:

```text
L = α · KL(Teacher, Student)  +  (1−α) · CE(y, Student)
Temperature T = 4~10 (일반적으로 5)  ·  α = 0.7~0.9  ·  Teacher는 well-trained 필수
```

우리의 배역: **Teacher = FP32 원본 모델**(Part 1), **Student = 수술 직후 slim_raw**(Part 3).
"큰 스승이 작은 제자의 재활을 돕는다" — 프루닝과 KD의 실전 조합입니다.

### Step 4-1. 온도(T)의 의미 — 왜 정답표(hard label)보다 나은가

같은 logit을 T=1과 T=5로 softmax하면 무엇이 달라지는지 먼저 봅니다.

In [ ]:
logits = torch.tensor([[3.0, 1.0, 0.2, -1.0]])       # 어떤 이미지에 대한 Teacher의 logit
labels4 = ["cat", "dog", "deer", "car"]

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
hard = [1.0, 0, 0, 0]
axes[0].bar(labels4, hard, color="#999999"); axes[0].set_title("hard label (정답표)\n'cat 100%'")
for ax, T in zip(axes[1:], [1, 5]):
    p = F.softmax(logits / T, dim=1)[0]
    ax.bar(labels4, p.tolist(), color="#4c72b0")
    ax.set_title(f"soft label (T={T})")
    for i, v in enumerate(p.tolist()):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
axes[0].set_ylabel("확률"); plt.tight_layout(); plt.show()

print("💡 T=1: 'cat 82%' — 거의 정답표와 비슷한 정보량")
print("   T=5: 'cat 37%, dog 25%, deer 21%...' — 'cat은 dog·deer와 닮았고 car와는 다르다'는")
print("   클래스 간 유사도 정보가 드러납니다. 이것이 dark knowledge —")
print("   정답표에는 없는, Teacher만 아는 지식입니다.")

### Step 4-2. KD loss — 이미 구현되어 있습니다

Part 3-4의 `finetune()` 함수에 `teacher`를 넘기면 KD 모드가 됩니다. 핵심 4줄만 다시 보면:

```python
soft = F.kl_div(F.log_softmax(s_logits / T, dim=1),      # Student의 soft 예측
                F.softmax(t_logits / T, dim=1),           # Teacher의 soft 정답
                reduction="batchmean") * (T * T)          # ★ T² 스케일
loss = alpha * soft + (1 - alpha) * F.cross_entropy(s_logits, y)
```

> **T²는 왜 곱하나?** softmax를 T로 나누면 기울기가 1/T²로 작아집니다.
> T²를 곱해 soft 항의 기울기 크기를 CE 항과 균형 맞추는 표준 보정입니다 (Hinton et al., 2015).

### Step 4-3. 대결 — 같은 출발점, CE vs KD

In [ ]:
slim_kd, acc_kd = finetune(slim_raw, teacher=model, T=5.0, alpha=0.8, tag="KD fine-tune (T=5, α=0.8)")

print()
print(f"{'재활 방법':<22}{'정확도':>9}")
print("-" * 32)
print(f"{'수술 직후 (재학습 X)':<22}{s_acc_raw:>8.2f}%")
print(f"{'CE fine-tune':<22}{acc_ce:>8.2f}%")
print(f"{'KD fine-tune':<22}{acc_kd:>8.2f}%")
print()
if acc_kd > acc_ce:
    print(f"→ KD가 CE보다 {acc_kd-acc_ce:+.2f}%p 우세 — dark knowledge의 값어치입니다.")
else:
    print("→ 이번 실행에선 KD 이득이 안 보이나요? 짧은 재학습·소형 모델에선 차이가 작을 수 있습니다.")
    print("  Teacher-Student 크기 격차가 클수록(교안: Student는 1/4~1/10), 학습이 길수록 KD가 유리해집니다.")
print("\n✏️ 직접 해보기: T=1(사실상 hard 위주), T=10, α=0.5로 바꿔 재실험 — 교안 가이드(T=4~10, α=0.7~0.9)의")
print("   범위를 벗어나면 어떻게 되는지 확인해 보세요.")

> **✅ Part 4 확인**
> - [ ] T가 클수록 클래스 간 유사도(dark knowledge)가 드러남을 시각화로 확인했다
> - [ ] KD loss의 두 항(soft KL + hard CE)과 T² 보정의 역할을 설명할 수 있다
> - [ ] 프루닝 후 재활에 KD를 조합하는 실전 패턴을 실행했다

---
# Part 5. 최종 조합 — 프루닝 + KD + 양자화 = 교안의 "10×"

교안 Day 2 도입부의 목표를 기억하시나요?

> **"모델 크기를 10× 이상 줄이고 정확도 저하를 최소화"** (학습 목표 02)
> BEFORE ResNet50 98MB → AFTER XWN 9.4MB (≈10×), 정확도 −1.3%p

우리 버전으로 재현합니다: **구조적 프루닝(2×) × 양자화(4×) ≈ 8~12×**.
KD로 재활한 `slim_kd`에 양자화 미니랩의 PTQ를 그대로 적용합니다.

### Step 5-1. slim_kd에 PTQ 적용 (양자화 미니랩 파이프라인 재사용)

In [ ]:
final = copy.deepcopy(slim_kd).eval()
final.fuse_model()                                        # Conv+BN+ReLU 융합
final.qconfig = tq.get_default_qconfig("fbgemm")
tq.prepare(final, inplace=True)
calib = torch.stack([train_set[i][0] for i in range(300)])
with torch.no_grad():
    for i in range(0, 300, 64):
        final(calib[i:i+64])                              # 캘리브레이션
tq.convert(final, inplace=True)                           # INT8 박제

final_acc, final_size, final_ms = evaluate(final, test_loader), model_size_kb(final), bench_ms(final)
print("3종 조합 완료: 구조적 프루닝 → KD 재활 → PTQ 양자화")

### Step 5-2. 최종 성적표 — Before / After

In [ ]:
print(f"{'':14}{'BEFORE (FP32 원본)':>18}{'AFTER (프루닝+KD+INT8)':>22}")
print("-" * 58)
print(f"{'정확도':<12}{fp32_acc:>17.2f}%{final_acc:>21.2f}%   ({final_acc-fp32_acc:+.2f}%p)")
print(f"{'크기':<13}{fp32_size:>16.0f}KB{final_size:>20.0f}KB   ({fp32_size/final_size:.1f}배 압축)")
print(f"{'지연':<13}{fp32_ms:>16.1f}ms{final_ms:>20.1f}ms   ({fp32_ms/final_ms:.1f}배 가속)")
print()
goal = fp32_size / final_size
print(f"{'🎉 교안 목표 10× 압축' if goal >= 10 else f'압축 {goal:.1f}× — keep 비율·양자화 조합을 조정해 10×에 도전!'}")
print()
print("💡 이 3단 조합(구조 절제 → 최적화 인지 재학습 → 정밀도 축소)이 바로")
print("   디퍼아이 XWN의 설계 사상입니다: '형태변환(Transform) + 프루닝으로 경량화,")
print("   정확도 저하는 QAT로 보완'(교안) — Day 2 본 실습에서 DDesignerAPI 한 번의 호출로 만납니다.")

---
# Part 6. 리포트 과제 & Day 2 연결

## 제출 과제 — `prune_kd_report.md`

**A. 최종 성적표** (본인 런타임 기준)

| 모델 | 정확도(%) | 크기(KB) | 지연(ms) |
| --- | --- | --- | --- |
| FP32 원본 | | | |
| 비구조 90% | | | |
| 구조 50% 수술 직후 | | | |
| 구조 50% + CE | | | — |
| 구조 50% + KD | | | — |
| 최종 (프루닝+KD+INT8) | | | |

**B. 분석 질문 (각 2~3문장)**
1. 비구조 90%의 크기·속도가 불변인 이유를 "dense 텐서의 shape" 관점에서 설명하시오.
2. 연쇄 수술에서 `w[:, prev]` 한 줄이 하는 일은 무엇이며, 이를 빠뜨리면 어떤 에러가 나는가? (✏️ 실제로 지워보고 확인!)
3. T=5의 soft label이 hard label보다 Student에게 유리한 이유를 "정보량" 관점에서 설명하시오.
4. BlackSwan의 Pruning Engine이 있으면 비구조적 프루닝도 실속 가속이 될 수 있는가? 가상 NPU 실습 도전과제(0-스킵 MAC)와 연결해 논하시오.

**C. 스크린샷**: sparsity-정확도 곡선, 채널 중요도 그래프, T=1 vs 5 분포, 최종 성적표

## 오늘 배운 것 ↔ 커리큘럼 대응표

| 이 실습 | 교안 / 본 실습 대응 |
| --- | --- |
| 비구조 90%의 배신 | "일반 NPU에서 속도 향상 X" — Lottery Ticket 가설의 실용적 한계 |
| 연쇄 수술 + 재학습 | 구조적 프루닝 "Fine-tuning 필수" · BlackSwan Pruning Engine |
| KD (T=5, α=0.8) | 교안 KD 실전 가이드 그대로 |
| 3종 조합 10× | Day 2 Before/After (98MB→9.4MB) · XWN 설계 사상 |
| `finetune(teacher=...)` | Day 2 2단계 학습(bsnet-t-o)의 "최적화 인지 재학습" 정신 |

## ✏️ 심화 도전 과제 (선택)

1. **Lottery Ticket 맛보기**: 프루닝으로 찾은 마스크를 유지한 채 살아남은 weight를 초기값으로 되돌려 재학습 — 같은 정확도에 도달하는가? (교안 "Lottery Ticket 가설")
2. **레이어 민감도 지도**: 한 블록씩만 keep=0.25로 강하게 잘라보며 정확도 하락 비교 — 어느 레이어가 프루닝에 민감한가?
3. **keep 비율 sweep**: 0.75/0.5/0.25로 구조적 프루닝 후 각각 KD 재활 — "최대 50%가 현실적"(교안)이 여러분 모델에서도 성립하는가?
4. **가상 NPU 연동**: 가상 NPU 실습의 `systolic_matmul`에 0-스킵을 구현하고, 구조 vs 비구조 프루닝된 weight로 유효 사이클 차이를 측정 — Pruning Engine의 원리 증명

---
수고하셨습니다! 🎉 이제 경량화 3종 세트(양자화·프루닝·KD)를 모두 손으로 다뤄봤습니다.
Day 2 본 실습의 XWN은 이 세 가지가 하나의 API로 통합된 것 — 여러분은 이미 그 내부를 알고 있습니다.
